In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
print("Day 4 libraries loaded successfully")

Day 4 libraries loaded successfully


In [6]:
df = pd.read_csv("../data/fraudlens_day3_features.csv")
print("Datset loaded successfully")
print("Shape",df.shape)

Datset loaded successfully
Shape (590540, 19)


In [7]:
print(df.head())
print(df.columns.tolist())

   TransactionDT  TransactionAmt  TransactionAmt_log  TransactionHour  \
0          86400            68.5            4.241327                0   
1          86401            29.0            3.401197                0   
2          86469            59.0            4.094345                0   
3          86499            50.0            3.931826                0   
4          86506            50.0            3.931826                0   

   TransactionDay ProductCD  card1  card2  card3       card4  card5   card6  \
0               1         W  13926    NaN  150.0    discover  142.0  credit   
1               1         W   2755  404.0  150.0  mastercard  102.0  credit   
2               1         W   4663  490.0  150.0        visa  166.0   debit   
3               1         W  18132  567.0  150.0  mastercard  117.0   debit   
4               1         H   4497  514.0  150.0  mastercard  102.0  credit   

   addr1  addr2 P_emaildomain R_emaildomain  EmailDomainMatch  MissingCount  \
0  315.

In [8]:
x = df.drop("isFraud",axis=1)
y = df["isFraud"]
print("Features",x.shape)
print("Target",y.shape)

Features (590540, 18)
Target (590540,)


In [9]:
print("Fraud distribution")
print(y.value_counts())
print("\nPercentage")
print(y.value_counts(normalize=True)*100)

Fraud distribution
isFraud
0    569877
1     20663
Name: count, dtype: int64

Percentage
isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64


In [11]:
categorical_columns = x.select_dtypes(
    include = ["object","category"]
).columns.tolist()
print("Categorical columns")
print(categorical_columns)

Categorical columns
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']


In [14]:
x = pd.get_dummies(
     x,
     columns = categorical_columns,
     drop_first = True
 )
print("Shape after encoing",x.shape)

Shape after coming (590540, 140)


In [15]:
x = x.replace([np.inf,-np.inf],np.nan)
print("Missng values before filling:",x.isnull().sum().sum())

Missng values before filling: 146169


In [18]:
imputer = SimpleImputer(strategy="median")
x_imputed = imputer.fit_transform(x)
print("Missing values handled successfully")
print("New Shape",x_imputed.shape)

Missing values handled successfully
New Shape (590540, 140)


In [19]:
x_train,x_test,y_train,y_test=train_test_split(
    x_imputed,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify=y
)
print("Training samples",x_train.shape[0])

print("Testing samples",x_test.shape[0])

Training samples 472432
Testing samples 118108


In [22]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)
print("Feature scalling completed")

Feature scalling completed


model_lr = LogisticRegression(
    max_iter = 1000,
    class_weight = "balanced",
    random_state = 42
)
model_ir.fit(x_train_scaled,y_train)
print("Logistic Regression trained successfully")

In [31]:
model_lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model_lr.fit(x_train_scaled, y_train)

print("Logistic Regression trained successfully")

Logistic Regression trained successfully


In [32]:
y_pred = model_lr.predict(x_test_scaled)
y_prob = model_lr.predict_proba(x_test_scaled)[:,1]
print("Predictions generated successfully")

Predictions generated successfully


In [33]:
accuracy = accuracy_score(y_test,y_pred)
precision = precision_score(y_test,y_pred)
recall = recall_score(y_test,y_pred)
f1 = f1_score(y_test,y_pred)
roc_auc = roc_auc_score(y_test,y_prob)
print("========= Logistic regression=========")
print("Accuracy:",round(accuracy,4))
print("Precision",round(precision,4))
print("Recall:",round(recall,4))
print("F1Score:",round(f1,4))
print("ROC-AUC:",round(roc_auc,4))

========= Logistic regression=========
Accuracy: 0.7673
Precision 0.0924
Recall: 0.6405
F1Score: 0.1615
ROC-AUC: 0.7684


In [34]:
cm = confusion_matrix(y_test,y_pred)
print("Confusion Matrix")
print(cm)

Confusion Matrix
[[87981 25994]
 [ 1486  2647]]


In [35]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.77      0.86    113975
           1       0.09      0.64      0.16      4133

    accuracy                           0.77    118108
   macro avg       0.54      0.71      0.51    118108
weighted avg       0.95      0.77      0.84    118108



In [36]:
import joblib
joblib.dump(model_lr,"../models/fraudlens_logistic_baseline.aok")
print("Baseline model saved successfully")

Baseline model saved successfully


In [39]:
joblib.dump(
    imputer,
    "../models/fraudlens_imputer.pkl"
)
joblib.dump(
    scaler,
    "../models/fraudlens_scaler.pkl"
)

print("Preprocessing objects saved successfully")

Preprocessing objects saved successfully


In [42]:
results = pd.DataFrame({
    "Model":["Logistic Regression"],
    "Accuracy":[accuracy],
    "Precision":[precision],
    "Recall":[recall],
    "F1 Score":[f1],
    "ROC-AUC":[roc_auc]
})

In [45]:
results.to_csv(
    "../data/day4_model_results.csv",
    index=False
)

print("Day 4 results saved successfully")

Day 4 results saved successfully
